# Multi-Modality Cell Segmentation with OneFormer
### Remote GPU Training Notebook (Google Colab)
Benchmark based on: *The Multi-modality Cell Segmentation Challenge: Towards Universal Solutions (Nature Methods 2024)*
Dataset: `MyDrive/DatasetCellSeg/` with `training/`, `tuning/`, and `testing/`.

## 1. Verify GPU Availability
Ensure you are running on a GPU runtime:
👉 **Exécution (Runtime) > Modifier le type d'exécution > Accélérateur matériel : GPU (T4 ou A100)**

In [ ]:
!nvidia-smi

## 2. Mount Google Drive
Mount your Google Drive to access `DatasetCellSeg` and permanently save checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repository & Install Dependencies

In [ ]:
import os

# Cloner votre dépôt GitHub CellOneFormer s'il n'est pas déjà présent
if not os.path.exists("/content/CellOneFormer"):
    !git clone https://github.com/BaptFontaine/CellOneFormer.git /content/CellOneFormer
else:
    # Mettre à jour avec les derniers commits si déjà présent
    !cd /content/CellOneFormer && git pull

# Se positionner dans le dossier du projet
%cd /content/CellOneFormer
%env PYTHONPATH=/content/CellOneFormer

# Installer toutes les dépendances
!pip install -r requirements.txt

## 4. Copy DatasetCellSeg to High-Speed Colab Local SSD
Copies `MyDrive/DatasetCellSeg` into `/content/CellOneFormer/data/raw/DatasetCellSeg` to eliminate network drive latency during training.

In [ ]:
import os
from pathlib import Path

DRIVE_DATASET_PATH = "/content/drive/MyDrive/DatasetCellSeg"
TARGET_RAW_DIR = "/content/CellOneFormer/data/raw/DatasetCellSeg"

print(f"Vérification de {DRIVE_DATASET_PATH}...")
if os.path.exists(DRIVE_DATASET_PATH):
    print("Dossiers détectés dans DatasetCellSeg :", os.listdir(DRIVE_DATASET_PATH))
    os.makedirs(TARGET_RAW_DIR, exist_ok=True)
    # Copie rapide vers le SSD local de Colab
    !cp -r "{DRIVE_DATASET_PATH}"/* "{TARGET_RAW_DIR}/"
    print(f"✅ Données copiées avec succès dans {TARGET_RAW_DIR}")
else:
    print(f"⚠️ Chemin introuvable : {DRIVE_DATASET_PATH}")
    print("Vérifiez l'orthographe exacte dans votre Google Drive.")

## 5. Tile & Preprocess the Dataset
Slices the large images from `training` and `tuning` into $512 \times 512$ patches with 64 px overlap.

In [ ]:
%cd /content/CellOneFormer
%env PYTHONPATH=/content/CellOneFormer
!python -c "import yaml; from src.training.train import prepare_tiled_data; cfg = yaml.safe_load(open('configs/config.yaml')); prepare_tiled_data(cfg)"

## 6. Launch Training on GPU
Trains OneFormer on the tiled patches with mixed precision (`fp16`). Checkpoints are saved in `checkpoints/`.

In [ ]:
%cd /content/CellOneFormer
%env PYTHONPATH=/content/CellOneFormer
!python src/training/train.py --config configs/config.yaml --device cuda

## 7. Backup Checkpoints to Google Drive
Saves the trained weights permanently into `MyDrive/cell_oneformer_checkpoints`.

In [ ]:
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/cell_oneformer_checkpoints"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

!cp -r /content/CellOneFormer/checkpoints/* "{DRIVE_BACKUP_DIR}/"
print(f"✅ Modèle sauvegardé avec succès sur Google Drive : {DRIVE_BACKUP_DIR}")

## 8. Inference & Full-Field Stitching on Testing Split
Runs inference on high-resolution images from `testing`, stitches overlapping tiles back together, and saves predictions.

In [ ]:
%cd /content/CellOneFormer
%env PYTHONPATH=/content/CellOneFormer
import glob

test_images = glob.glob("data/raw/DatasetCellSeg/testing/*.*") + glob.glob("data/raw/DatasetCellSeg/testing/images/*.*")
if len(test_images) > 0:
    sample_test = test_images[0]
    print(f"Test d'inférence sur : {sample_test}")
    !python src/inference/predict.py \
        --image "{sample_test}" \
        --weights checkpoints/oneformer_cells_best.pt \
        --output-dir predictions
else:
    print("Aucune image trouvée dans testing/")